In [1]:
import os
os.chdir('../')
%pwd

'/Users/kiranprasadjp/Documents/Pros/HHMI_Janelia_AI_Engineer'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelBuildConfig:
    root_dir: Path
    model_name: str
    repo: str
    mw: Path
    hug_mw: str
    HF_save: Path
    out: Path
    param_dd: tuple

In [6]:
from src.task2 import logger
from src.task2.constants import *
from src.task2.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_build_config(self) -> ModelBuildConfig:
        config = self.config.model_build

        create_directories([config.root_dir])

        model_build_config = ModelBuildConfig(
            root_dir=Path(config.root_dir),
            model_name= str(config.model_name),
            repo=str(config.REPO),
            mw=Path(config.M_weight),
            hug_mw=str(config.Hug_MW),
            HF_save=Path(config.Hug_save),
            out= Path(config.OUTPUT),
            param_dd= tuple(self.params.input_size)
        )

        return model_build_config

In [9]:
import os
import torch
import multiprocessing
import torch.nn.functional as F
import numpy as np
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

In [10]:
from torchinfo import summary
import datetime

In [ ]:
class ModelBuild:
    def __init__(self, config:ModelBuildConfig ):
        self.config=config
    
    def sysConfig(self):
        logger.info("🔍 --- Project Diagnostic ---")

        # 1. Multiprocessing Check
        # Colab uses Linux, which defaults to 'fork'. 
        # For CUDA, 'spawn' is technically safer to avoid deadlocks.
        logger.info("[MULTIPROCESSING]")
        try:
            start_method = multiprocessing.get_start_method()
            logger.info(f"  Start method : {start_method}")
            cpu_count = multiprocessing.cpu_count()
            logger.info(f"  CPU cores    : {cpu_count} logical")
        except Exception as e:
            logger.error(f"  Multiprocessing check failed: {e}")

        # 2. Hardware & Backend Check
        logger.info("[DEVICE & BACKEND]")
        
        # NVIDIA CUDA (Colab Standard)
        if torch.cuda.is_available():
            device = torch.device("cuda")
            prop = torch.cuda.get_device_properties(0)
            
            logger.info(f"  Backend      ⚙️ : CUDA (NVIDIA)")
            logger.info(f"  GPU Model    : {prop.name}")
            logger.info(f"  VRAM Total   : {prop.total_memory / 1e9:.1f} GB")
            
            # Check for 'Compute Capability' (DINOv2 runs best on 7.0+)
            cc = f"{prop.major}.{prop.minor}"
            logger.info(f"  Compute Cap  : {cc}")
            
            # Check Memory Fragmentation
            vram_reserved = torch.cuda.memory_reserved(0) / 1e9
            vram_allocated = torch.cuda.memory_allocated(0) / 1e9
            logger.info(f"  VRAM Reserved: {vram_reserved:.1f} GB")
            logger.info(f"  VRAM Active  : {vram_allocated:.1f} GB")

            # Smoke Test
            try:
                test_tensor = torch.zeros((100, 100), device=device)
                logger.info("  Smoke test 💨 : CUDA Tensor creation OK")
                del test_tensor # Clean up immediately
            except Exception as e:
                logger.warning(f"  Smoke test 💨 : FAILED — {e}")

        # Apple Silicon (For when you run locally)
        elif torch.backends.mps.is_available():
            device= torch.device('mps')
            logger.info("  Backend      ⚙️ : Apple Silicon (MPS)")
            logger.info("  Status       ✅: OK")

        else:
            logger.warning("  Backend      ⚙️ : CPU only — No Accelerator Found")

        # 3. Environment Context (Colab Specific)
        if 'COLAB_GPU' in os.environ:
            logger.info("  Environment  🌐: Google Colab detected")

        logger.info(f'Loading device and storing: {device}')
        
        return device

    def loadFromHuggingFace(self,device):
        model = AutoModel.from_pretrained(
            self.config.hug_mw, 
            cache_dir=self.config.HF_save,
            device_map="auto"
        )

        model_stats = summary(
            model, 
            input_size=tuple(self.config.param_dd),
            col_names=["input_size", "output_size", "num_params", "mult_adds"],
            col_width=20,
            device=device
            )

        with open(self.config.out, 'a') as f:
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            f.write(f"\n--- Analysis Run: {timestamp} ---\n Hugging Face Loaded \n")
            f.write(f"Model Summary for {self.config.model_name}\n")
            f.write(f"-"*30+"\n")
            f.write(f"{model_stats}\n")
            f.write(f"-"*30+"\n")
            f.write(f"==== Model BluePrint ====\n")
            f.write(f"{model}\n")
            f.write(f"-"*30+"\n")

        return model
    
    def loadFromLocal(self,device):
        model = torch.hub.load(
            self.config.repo, 
            self.config.model_name, 
            pretrained=False
        ).to(device)

        checkpoint = torch.load(self.config.mw, map_location=device)
        state_dict = {k.replace('module.', ''): v for k, v in checkpoint.items()}
        model.load_state_dict(state_dict)
        model.to(device).eval()

        logger.info(f"DINOv3 {self.config.model_name} Weights successfully injected from flat checkpoint.")


        model_stats = summary(
            model,
            input_size=tuple(self.config.param_dd),
            col_names=["input_size", "output_size", "num_params", "mult_adds"],
            col_width=20,
            device=device
            )

        # checkpoint = torch.load(self.config.mw, map_location=device)
        with open(self.config.out, 'a') as f:
            timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            f.write(f"\n--- Analysis Run: {timestamp} ---\n")
            f.write(f"==== Model BluePrint LOCAL LOAD ====\n")
            f.write(f"{model}\n")
            f.write(f"-"*30+"\n")
            f.write(f"Model Summary for {self.config.model_name}\n")
            f.write(f"-"*30+"\n")
            f.write(f"{model_stats}\n")
            f.write(f"-"*30+"\n")

        return model
    
    def get_dense_embeddings(self,device,model):
        





        

In [15]:
try:
    config = ConfigurationManager()
    model_enc_config = config.get_model_build_config()
    diagnosis = ModelBuild(config=model_enc_config)
    device=diagnosis.sysConfig()
    diagnosis.loadFromHuggingFace(device=device)
    diagnosis.loadFromLocal(device=device)
    
except Exception as e:
    raise e

[2026-04-03 21:07:53,678: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-03 21:07:53,681: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-03 21:07:53,681: INFO: common: created directory at: artifacts]
[2026-04-03 21:07:53,682: INFO: common: created directory at: artifacts/models]
[2026-04-03 21:07:53,683: INFO: 2406038666: 🔍 --- Project Diagnostic ---]
[2026-04-03 21:07:53,684: INFO: 2406038666: [MULTIPROCESSING]]
[2026-04-03 21:07:53,685: INFO: 2406038666:   Start method : spawn]
[2026-04-03 21:07:53,685: INFO: 2406038666:   CPU cores    : 10 logical]
[2026-04-03 21:07:53,686: INFO: 2406038666: [DEVICE & BACKEND]]
[2026-04-03 21:07:53,689: INFO: 2406038666:   Backend      ⚙️ : Apple Silicon (MPS)]
[2026-04-03 21:07:53,689: INFO: 2406038666:   Status       ✅: OK]
[2026-04-03 21:07:53,690: INFO: 2406038666: Loading device and storing: mps]
[2026-04-03 21:07:53,848: INFO: _client: HTTP Request: HEAD https://huggingface.co/facebook/dinov3-vit

Loading weights: 100%|██████████| 235/235 [00:00<00:00, 467.28it/s]


[2026-04-03 21:07:54,554: INFO: vision_transformer: using base=100 for rope new]
[2026-04-03 21:07:54,555: INFO: vision_transformer: using min_period=None for rope new]
[2026-04-03 21:07:54,555: INFO: vision_transformer: using max_period=None for rope new]
[2026-04-03 21:07:54,556: INFO: vision_transformer: using normalize_coords=separate for rope new]
[2026-04-03 21:07:54,556: INFO: vision_transformer: using shift_coords=None for rope new]
[2026-04-03 21:07:54,557: INFO: vision_transformer: using rescale_coords=2 for rope new]
[2026-04-03 21:07:54,557: INFO: vision_transformer: using jitter_coords=None for rope new]
[2026-04-03 21:07:54,558: INFO: vision_transformer: using dtype=fp32 for rope new]
[2026-04-03 21:07:54,559: INFO: vision_transformer: using swiglu layer as FFN]


Using cache found in /Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov3_main


In [ ]:
def loadFromHuggingFace(self,device):
        model = AutoModel.from_pretrained(
            self.config.hug_mw, 
            cache_dir=self.config.HF_save,
            device_map="auto"
        )

        model_stats = summary(
            model, 
            input_size=tuple(self.config.param_dd),
            col_names=["input_size", "output_size", "num_params", "mult_adds"],
            col_width=20,
            device=device
            )

        return model


    

In [33]:
import torch

# 1. Initialize the model from the GitHub source
# This automatically handles the 'DNA' of the model (layers, attention, etc.)

model = torch.hub.load(
    'facebookresearch/dinov3', 
    'dinov3_vits16plus', 
    pretrained=False
)

# 2. Configure for your MacBook M2 Pro GPU (MPS)
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model.to(device)

# 3. Load your specific LVD-1689M weights
checkpoint = torch.load('artifacts/models/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth', map_location=device)
state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
# Sometimes it's nested in 'teacher' for the distilled versions
if 'teacher' in state_dict:
    state_dict = state_dict['teacher']

# Remove 'module.' prefix if the weights were saved via DistributedDataParallel
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict)

model.eval()
print(f"DINOv3 S+ (29M) active on {device}")

Using cache found in /Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov3_main


[2026-04-03 20:40:11,995: INFO: vision_transformer: using base=100 for rope new]
[2026-04-03 20:40:11,996: INFO: vision_transformer: using min_period=None for rope new]
[2026-04-03 20:40:11,996: INFO: vision_transformer: using max_period=None for rope new]
[2026-04-03 20:40:11,997: INFO: vision_transformer: using normalize_coords=separate for rope new]
[2026-04-03 20:40:11,997: INFO: vision_transformer: using shift_coords=None for rope new]
[2026-04-03 20:40:11,997: INFO: vision_transformer: using rescale_coords=2 for rope new]
[2026-04-03 20:40:11,997: INFO: vision_transformer: using jitter_coords=None for rope new]
[2026-04-03 20:40:11,998: INFO: vision_transformer: using dtype=fp32 for rope new]
[2026-04-03 20:40:11,998: INFO: vision_transformer: using swiglu layer as FFN]
DINOv3 S+ (29M) active on mps


In [14]:
params_dict = model.state_dict()

# 2. To see the names of the layers (useful for debugging):
print(params_dict.keys())

odict_keys(['cls_token', 'storage_tokens', 'mask_token', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'rope_embed.periods', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.qkv.bias_mask', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.ls1.gamma', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.w1.weight', 'blocks.0.mlp.w1.bias', 'blocks.0.mlp.w2.weight', 'blocks.0.mlp.w2.bias', 'blocks.0.mlp.w3.weight', 'blocks.0.mlp.w3.bias', 'blocks.0.ls2.gamma', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.qkv.bias_mask', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.1.ls1.gamma', 'blocks.1.norm2.weight', 'blocks.1.norm2.bias', 'blocks.1.mlp.w1.weight', 'blocks.1.mlp.w1.bias', 'blocks.1.mlp.w2.weight', 'blocks.1.mlp.w2.bias', 'blocks.1.mlp.w3.weight', 'blocks.1.mlp.w3.bias', 'blocks.1.ls2.gamma', 'blocks

In [ ]:

batch_size = 1
summary(model, input_size=(batch_size, 3, 448, 448))

Layer (type:depth-idx)                        Output Shape              Param #
DinoVisionTransformer                         [1, 384]                  2,304
├─PatchEmbed: 1-1                             [1, 28, 28, 384]          --
│    └─Conv2d: 2-1                            [1, 384, 28, 28]          295,296
│    └─Identity: 2-2                          [1, 784, 384]             --
├─RopePositionEmbedding: 1-2                  [784, 64]                 --
├─ModuleList: 1-25                            --                        (recursive)
│    └─SelfAttentionBlock: 2-3                [1, 789, 384]             --
│    │    └─LayerNorm: 3-1                    [1, 789, 384]             768
│    │    └─SelfAttention: 3-2                [1, 789, 384]             591,360
│    │    └─LayerScale: 3-3                   [1, 789, 384]             384
│    │    └─LayerNorm: 3-4                    [1, 789, 384]             768
│    │    └─SwiGLUFFN: 3-5                    [1, 789, 384]           

In [ ]:
from torchinfo import summary
import torch

input_size = (1, 3, 448, 448)

model_stats = summary(
    model, 
    input_size=input_size,
    col_names=["input_size", "output_size", "num_params", "mult_adds"],
    col_width=20,
    device="mps" 
)

print(model_stats)

Layer (type:depth-idx)                             Input Shape          Output Shape         Param #              Mult-Adds
DINOv3ViTModel                                     [1, 3, 448, 448]     [1, 384]             --                   --
├─DINOv3ViTEmbeddings: 1-1                         [1, 3, 448, 448]     [1, 789, 384]        2,304                --
│    └─Conv2d: 2-1                                 [1, 3, 448, 448]     [1, 384, 28, 28]     295,296              231,512,064
├─DINOv3ViTRopePositionEmbedding: 1-2              [1, 3, 448, 448]     [784, 64]            --                   --
├─DINOv3ViTEncoder: 1-3                            [1, 789, 384]        [1, 789, 384]        --                   --
│    └─ModuleList: 2-2                             --                   --                   --                   --
│    │    └─DINOv3ViTLayer: 3-1                    [1, 789, 384]        [1, 789, 384]        2,366,208            2,365,440
│    │    └─DINOv3ViTLayer: 3-2          

In [1]:
import torch
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image(url)

pretrained_model_name = "facebook/dinov3-vits16plus-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
)

inputs = processor(images=image, return_tensors="pt").to(model.device)
with torch.inference_mode():
    outputs = model(**inputs)

pooled_output = outputs.pooler_output
print("Pooled output shape:", pooled_output.shape)


Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

Pooled output shape: torch.Size([1, 384])


In [ ]:
import torch

# 1. Initialize the 'Skeleton' (Empty)
# We set pretrained=False because we already have the file locally
model = torch.hub.load(
    'facebookresearch/dinov3', 
    'dinov3_vits16plus', 
    pretrained=False
)
device = torch.device("mps")
checkpoint_path = "artifacts/models/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth"

# Map to MPS for M2 Pro speed
checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.to(device).eval()

print("DINOv3 S+ Gated Model: Successfully Loaded.")

Using cache found in /Users/kiranprasadjp/.cache/torch/hub/facebookresearch_dinov3_main
/Users/kiranprasadjp/Documents/Pros/HHMI_Janelia_AI_Engineer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DINOv3 S+ Gated Model: Successfully Loaded.


In [6]:
# 1. Get the list of all "Labels"
labels = list(state_dict.keys())

# 2. Print the first 10 to see the naming convention
print("--- First 10 Layer Labels ---")
for label in labels[:]:
    print(label)

# 3. Print the total count
print(f"\nTotal Number of Layers: {len(labels)}")

--- First 10 Layer Labels ---
cls_token
storage_tokens
mask_token
patch_embed.proj.weight
patch_embed.proj.bias
rope_embed.periods
blocks.0.norm1.weight
blocks.0.norm1.bias
blocks.0.attn.qkv.weight
blocks.0.attn.qkv.bias
blocks.0.attn.qkv.bias_mask
blocks.0.attn.proj.weight
blocks.0.attn.proj.bias
blocks.0.ls1.gamma
blocks.0.norm2.weight
blocks.0.norm2.bias
blocks.0.mlp.w1.weight
blocks.0.mlp.w1.bias
blocks.0.mlp.w2.weight
blocks.0.mlp.w2.bias
blocks.0.mlp.w3.weight
blocks.0.mlp.w3.bias
blocks.0.ls2.gamma
blocks.1.norm1.weight
blocks.1.norm1.bias
blocks.1.attn.qkv.weight
blocks.1.attn.qkv.bias
blocks.1.attn.qkv.bias_mask
blocks.1.attn.proj.weight
blocks.1.attn.proj.bias
blocks.1.ls1.gamma
blocks.1.norm2.weight
blocks.1.norm2.bias
blocks.1.mlp.w1.weight
blocks.1.mlp.w1.bias
blocks.1.mlp.w2.weight
blocks.1.mlp.w2.bias
blocks.1.mlp.w3.weight
blocks.1.mlp.w3.bias
blocks.1.ls2.gamma
blocks.2.norm1.weight
blocks.2.norm1.bias
blocks.2.attn.qkv.weight
blocks.2.attn.qkv.bias
blocks.2.attn.qkv.b

In [7]:
# This looks at the first word before the dot (e.g., 'blocks', 'patch_embed')
folders = sorted(list(set([k.split('.')[0] for k in state_dict.keys()])))

print("--- Top-Level Model Sections ('Folders') ---")
for folder in folders:
    # Count how many parameters are in each section
    count = len([k for k in state_dict.keys() if k.startswith(folder)])
    print(f"Folder: {folder:<15} | Layers: {count}")

--- Top-Level Model Sections ('Folders') ---
Folder: blocks          | Layers: 204
Folder: cls_token       | Layers: 1
Folder: mask_token      | Layers: 1
Folder: norm            | Layers: 2
Folder: patch_embed     | Layers: 2
Folder: rope_embed      | Layers: 1
Folder: storage_tokens  | Layers: 1


In [2]:
# 1. See the main 'folders' in the bundle
print("--- Checkpoint Bundle Keys ---")
print(checkpoint.keys())

# 2. Check the version or training progress if available
if 'epoch' in checkpoint:
    print(f"Model was trained for {checkpoint['epoch']} epochs.")
    
if 'arch' in checkpoint:
    print(f"Architecture defined in bundle: {checkpoint['arch']}")

--- Checkpoint Bundle Keys ---


NameError: name 'checkpoint' is not defined